# Creating a More Cloud-Native METAR Format
---

## Overview
   
Within this notebook, we will create an interactive visualization of the latest METAR data across all stations. We will use the following libraries for our visualizations:

1. [Xarray](https://docs.xarray.dev/en/stable/)

## Prerequisites
| Concepts | Importance | Notes |
| --- | --- | --- |
| [Zarr](https://zarr.readthedocs.io/en/stable/) | Required | Cloud Computing |

- **Time to learn**: 10 minutes
---

In [25]:
import duckdb
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime, timedelta, timezone
import hvplot
import hvplot.pandas
import holoviews as hv
from palettable.colorbrewer.diverging import BrBG_10
import xarray as xr

In [3]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import shapely

from lonboard import Map, ScatterplotLayer
from lonboard.colormap import apply_continuous_cmap
from lonboard.controls import MultiRangeSlider
from lonboard.layer_extension import DataFilterExtension

We will just be working with the past years worth of data, so the parquet file can be found here.

In [7]:
url = 'https://data.source.coop/dynamical/asos-parquet/year=2026/data.parquet'

## Get the past three days of data
This query will request just the latest hours worth of data across all stations.

In [10]:
time_1 = datetime.now()
print(f"end time: {time_1}")
time_0 = datetime.now() - timedelta(hours=72)
print(f"start time: {time_0}")

end time: 2026-06-18 15:58:59.289439
start time: 2026-06-15 15:58:59.289579


# Query the database for all data between those timestamps

In [11]:
df = duckdb.execute("""
    SELECT *
    FROM read_parquet($1, hive_partitioning=true)
    WHERE 
---      country = 'US' AND
    valid BETWEEN $2 AND $3
    ORDER BY country
""", [url, time_0, time_1]).fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Print the first couple columns of data to get an understanding of the structure

In [14]:
df.head(2)

,station,valid,longitude,latitude,tmpf,tmpc,dwpf,dwpc,relh,drct,...,state,geometry,name,elevation,country,county,wfo,tzname,bbox,year
0,ANYN,2026-06-16 08:00:00+00:00,166.9191,-0.5475,80.6,27.0,75.2,24.0,83.70,240.0,...,AU,"[1, 1, 0, 0, 0, 29, 56, 103, 68, 105, 221, 100...",Nauru,7.0,AU,NaN,NaN,Pacific/Nauru,"{'xmin': 166.9191, 'ymin': -0.5475, 'xmax': 16...",2026
1,NZQN,2026-06-15 16:00:00+00:00,168.7392,-45.0211,39.2,4.0,37.4,3.0,93.19,50.0,...,AU,"[1, 1, 0, 0, 0, 129, 38, 194, 134, 167, 23, 10...",Queenstown,356.0,AU,NaN,NaN,Pacific/Auckland,"{'xmin': 168.7392, 'ymin': -45.0211, 'xmax': 1...",2026


## Understanding the data columns

The documentation can be found here for the individual columns:

https://github.com/dynamical-org/asos-parquet#key-fields

Print the columns as follows:

In [15]:
df.columns

Index(['station', 'valid', 'longitude', 'latitude', 'tmpf', 'tmpc', 'dwpf',
       'dwpc', 'relh', 'drct', 'sknt', 'gust', 'alti', 'mslp', 'vsby', 'p01i',
       'p01m', 'state', 'geometry', 'name', 'elevation', 'country', 'county',
       'wfo', 'tzname', 'bbox', 'year'],
      dtype='str')

Now create bind descriptions to each of the columns

In [16]:
variables = {
    'tmpf': "Temperature in Fahrenheit",
    'tmpc': "Temperature in Celsius",
    'dwpf': "Dewpoint in Fahrenheit",
    'dwpc': "Dewpoint in Celsius", 
    'relh': "Relative humidity in percent", 
    'drct': "Wind direction in degrees", 
    'sknt': "Wind speed in knots", 
    'gust': "Wind gusts in knots", 
    'alti': "Altimeter setting in inches of mercury", 
    'mslp': "Mean sea level pressure", 
    'vsby': "Visibility in statute miles", 
    'p01i': "P01I", # 1-hour precipitation inches
    'p01m': "P01M" # 1-hour precipitation mm
}

## Plot Timeseries data from a station local to Boulder, Colorado
Select all the data from the Boulder station, "BDU" and plot it.

In [17]:
df_bdu = df.loc[df['station'] == "BDU"]
df_bdu.head(2)

,station,valid,longitude,latitude,tmpf,tmpc,dwpf,dwpc,relh,drct,...,state,geometry,name,elevation,country,county,wfo,tzname,bbox,year
419802,BDU,2026-06-15 16:15:00+00:00,-105.2258,40.0394,66.2,19.0,41.0,5.0,39.73,340.0,...,CO,"[1, 1, 0, 0, 0, 245, 219, 215, 129, 115, 78, 9...",Boulder,1612.0,US,Boulder,BOU,America/Denver,"{'xmin': -105.2258, 'ymin': 40.0394, 'xmax': -...",2026
419803,BDU,2026-06-15 16:35:00+00:00,-105.2258,40.0394,68.0,20.0,41.0,5.0,37.34,170.0,...,CO,"[1, 1, 0, 0, 0, 245, 219, 215, 129, 115, 78, 9...",Boulder,1612.0,US,Boulder,BOU,America/Denver,"{'xmin': -105.2258, 'ymin': 40.0394, 'xmax': -...",2026


In [18]:
df_select = pd.DataFrame({
    'date': df_bdu['valid'],
    'tmpc': df_bdu['tmpc'],
    'dwpc': df_bdu['dwpc'],
    'relh': df_bdu['relh'],
    'drct': df_bdu['drct'],
    'sknt': df_bdu['sknt'],
    'alti': df_bdu['alti'],
}).set_index('date')

# print the head of the dataframe
df_select.head(2)

,tmpc,dwpc,relh,drct,sknt,alti
date,,,,,,
2026-06-15 16:15:00+00:00,19.0,5.0,39.73,340.0,3.0,30.18
2026-06-15 16:35:00+00:00,20.0,5.0,37.34,170.0,3.0,30.18


Iterate through some variables and compose an interactive plot of the past three days

In [20]:
hv.Layout(
    [df_select[i].hvplot.line(title=f"{variables[i]}", width=400) for i in [
        'tmpc',
        'dwpc',
        'relh',
        'drct',
        'sknt',
        'alti'
    ]]
).cols(2)

:Layout
   .Curve.Tmpc :Curve   [date]   (tmpc)
   .Curve.Dwpc :Curve   [date]   (dwpc)
   .Curve.Relh :Curve   [date]   (relh)
   .Curve.Drct :Curve   [date]   (drct)
   .Curve.Sknt :Curve   [date]   (sknt)
   .Curve.Alti :Curve   [date]   (alti)

## Antipatterns

There are antipatterns in the cloud computing scheme
 - compression
 - partitioning

We would like to access the data using a more pythonic approach.

These can be fixed using Xarray's DataTree implementation.

In [28]:
df.head(2)

,station,valid,longitude,latitude,tmpf,tmpc,dwpf,dwpc,relh,drct,...,state,geometry,name,elevation,country,county,wfo,tzname,bbox,year
0,ANYN,2026-06-16 08:00:00+00:00,166.9191,-0.5475,80.6,27.0,75.2,24.0,83.70,240.0,...,AU,"[1, 1, 0, 0, 0, 29, 56, 103, 68, 105, 221, 100...",Nauru,7.0,AU,NaN,NaN,Pacific/Nauru,"{'xmin': 166.9191, 'ymin': -0.5475, 'xmax': 16...",2026
1,NZQN,2026-06-15 16:00:00+00:00,168.7392,-45.0211,39.2,4.0,37.4,3.0,93.19,50.0,...,AU,"[1, 1, 0, 0, 0, 129, 38, 194, 134, 167, 23, 10...",Queenstown,356.0,AU,NaN,NaN,Pacific/Auckland,"{'xmin': 168.7392, 'ymin': -45.0211, 'xmax': 1...",2026


In [61]:
# assemble all of the data for one station
df_bdu = df.loc[df['station'] == "BDU"]
df_bdu_select = pd.DataFrame({
    'date': df_bdu['valid'],
    'latitude': df_bdu['latitude'],
    'longitude': df_bdu['longitude'],
    'tmpf': df_bdu['tmpf'],
    'dwpf': df_bdu['dwpf'],
    'relh': df_bdu['relh'],
    'drct': df_bdu['drct'],
    'sknt': df_bdu['sknt'],
    'gust': df_bdu['gust'],
    'alti': df_bdu['alti'],
    # 'mslp': df_bdu['mslp'],
    'vsby': df_bdu['vsby'],
    'p01i': df_bdu['p01i'],
}).set_index('date')

df_bdu_select.head(10)

,latitude,longitude,tmpf,dwpf,relh,drct,sknt,gust,alti,vsby,p01i
date,,,,,,,,,,,
2026-06-15 16:15:00+00:00,40.0394,-105.2258,66.2,41.0,39.73,340.0,3.0,NaN,30.18,10.0,0.0
2026-06-15 16:35:00+00:00,40.0394,-105.2258,68.0,41.0,37.34,170.0,3.0,NaN,30.18,10.0,0.0
2026-06-15 16:55:00+00:00,40.0394,-105.2258,68.0,41.0,37.34,20.0,4.0,NaN,30.18,10.0,0.0
2026-06-15 17:15:00+00:00,40.0394,-105.2258,69.8,41.0,35.11,130.0,3.0,NaN,30.18,10.0,0.0
2026-06-15 17:35:00+00:00,40.0394,-105.2258,71.6,42.8,35.40,120.0,5.0,NaN,30.18,10.0,0.0
2026-06-15 17:55:00+00:00,40.0394,-105.2258,71.6,42.8,35.40,60.0,4.0,NaN,30.17,10.0,0.0
2026-06-15 18:15:00+00:00,40.0394,-105.2258,71.6,42.8,35.40,0.0,0.0,NaN,30.17,10.0,0.0
2026-06-15 18:35:00+00:00,40.0394,-105.2258,73.4,42.8,33.31,100.0,4.0,NaN,30.16,10.0,0.0
2026-06-15 18:55:00+00:00,40.0394,-105.2258,75.2,41.0,29.26,160.0,5.0,NaN,30.15,10.0,0.0


In [78]:
latitude = df_bdu['latitude'].iloc[0]
longitude = df_bdu['longitude'].iloc[0]
elevation = df_bdu['elevation'].iloc[0]
print(f"Latitude: {latitude}, Longitude: {longitude}, Elevation: {elevation}")

Latitude: 40.0394, Longitude: -105.2258, Elevation: 1612.0


In [62]:
country = df_bdu['country'].iloc[0]
tzname = df_bdu['tzname'].iloc[0]
print(f"Country: {country} and time zone name: {tzname}")

Country: US and time zone name: America/Denver


In [63]:
# create an xarray dataarra for each variable
time = df_bdu_select.index

latitude = df_bdu_select['latitude'].values
longitude = df_bdu_select['longitude'].values

temperature = df_bdu_select['tmpf'].values
dew_point = df_bdu['dwpf'].values
relative_humidity = df_bdu['relh'].values
wind_direction = df_bdu['drct'].values
wind_speed = df_bdu['sknt'].values
wind_gust = df_bdu['gust'].values
pressure = df_bdu['alti'].values
# sea_level_pressure = df_bdu['mslp'].values
visibility = df_bdu['vsby'].values
one_hour_precipitation = df_bdu['p01i'].values

In [64]:
time_da = xr.DataArray(
    data=time,
    dims=("time"),
    name="time",
    attrs=dict(
        long_name="Timestamp of each ping",
        standard_name="time",
    )
)
# time_da

In [72]:
temperature_da = xr.DataArray(
    data=temperature,
    dims=["time"],
    coords=dict(
        time=time_da,
    ),
    attrs=dict(
        long_name="Temperature in Fahrenheit",
        standard_name="Temperature",
        units="degF",
    ),
)
# temperature_da
dew_point_da = xr.DataArray(
    data=dew_point,
    dims=["time"],
    coords=dict(time=time_da),
    attrs=dict(
        long_name="Dewpoint in Fahrenheit",
        standard_name="Dewpoint",
        units="degF",
    ),
)
relative_humidity_da = xr.DataArray(
    data=relative_humidity,
    dims=["time"],
    coords=dict(time=time_da),
    attrs=dict(
        long_name="Relative humidity in Percent",
        standard_name="Relative Humidity",
        units="percent",
    ),
)
wind_direction_da = xr.DataArray(
    data=wind_direction,
    dims=["time"],
    coords=dict(time=time_da),
    attrs=dict(
        long_name="Wind Direction in Degrees",
        standard_name="Wind Direction",
        units="degrees",
    ),
)
wind_speed_da = xr.DataArray(
    data=wind_speed,
    dims=["time"],
    coords=dict(time=time_da),
    attrs=dict(
        long_name="Wind Speed in Knots",
        standard_name="Wind Speed",
        units="knots",
    ),
)
wind_gust_da = xr.DataArray(
    data=wind_gust,
    dims=["time"],
    coords=dict(time=time_da),
    attrs=dict(
        long_name="Wind Gust in Knots",
        standard_name="Wind Gust",
        units="knots",
    ),
)
pressure_da = xr.DataArray(
    data=pressure,
    dims=["time"],
    coords=dict(time=time_da),
    attrs=dict(
        long_name="Pressure in inHg",
        standard_name="Pressure",
        units="inHg",
    ),
)
visibility_da = xr.DataArray(
    data=visibility,
    dims=["time"],
    coords=dict(time=time_da),
    attrs=dict(
        long_name="Visibility in Miles",
        standard_name="Visibility",
        units="miles",
    ),
)
one_hour_precipitation_da = xr.DataArray(
    data=one_hour_precipitation,
    dims=["time"],
    coords=dict(time=time_da),
    attrs=dict(
        long_name="One Hour Precipitation in Inches",
        standard_name="Precipitation",
        units="inches",
    ),
)
# one_hour_precipitation_da

In [79]:
ds = xr.Dataset(
    data_vars=dict(
        temperature=temperature_da,
        dew_point=dew_point_da,
        relative_humidity=relative_humidity_da,
        wind_direction=wind_direction_da,
        wind_speed=wind_speed_da,
        wind_gust=wind_gust_da,
        pressure=pressure_da,
        visibility=visibility_da,
        precipitation=one_hour_precipitation_da,
    ),
    coords=dict(
        time=time,
    ),
    attrs=dict(
        country=country,
        timezone=tzname,
        latitude=latitude,
        longitude=longitude,
        elevation=elevation
    ),
)
ds

<xarray.Dataset> Size: 17kB
Dimensions:            (date: 213, time: 213)
Coordinates:
    time               (date) datetime64[us, UTC] 2kB 2026-06-15 16:15:00+00:...
Dimensions without coordinates: date
Data variables:
    temperature        (time) float64 2kB 66.2 68.0 68.0 69.8 ... 64.4 66.2 66.2
    dew_point          (time) float64 2kB 41.0 41.0 41.0 41.0 ... 32.0 32.0 32.0
    relative_humidity  (time) float64 2kB 39.73 37.34 37.34 ... 27.84 27.84
    wind_direction     (time) float64 2kB 340.0 170.0 20.0 ... 360.0 80.0 0.0
    wind_speed         (time) float64 2kB 3.0 3.0 4.0 3.0 ... 4.0 3.0 3.0 0.0
    wind_gust          (time) float64 2kB nan nan nan nan ... nan nan nan nan
    pressure           (time) float64 2kB 30.18 30.18 30.18 ... 30.07 30.06
    visibility         (time) float64 2kB 10.0 10.0 10.0 10.0 ... 10.0 10.0 10.0
    precipitation      (time) float64 2kB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
Attributes:
    country:    US
    timezone:   America/Denver
    latitude:   40.0394
    longitude:  -105.2258
    elevation:  1612.0

TODO:
 - read in the data w duckdb (done)
 - save parquet file (done)
 - open and read a single stations worth of data (done)
 - create a dataset(done)
    - features,
    - lat, lon
    - add attributes for each variable
    - save ds (done)
 - iterate and do for all
 - create xarray datatree
 - attach datasets to datatree
 - write to local file system
 - get size
 - show opening & plotting timeseries

## References

1. [GeoPandas](https://geopandas.org)
1. [EPSG](https://epsg.io)

## What's next?
Expanding on the plotting capability to more interactively visualize the wind 'u' and 'v' components.